# Deltid personalegrupper

In [104]:
from __future__ import annotations

import requests

url = "https://www.krl.dk/sirka/sirkaApi/tableApi"

# KRL payload (outputFormat='json').
# Returnerer rækker med koder: overenskomst/stilling/klassificering + nøgletal pr. måned.
# NOTE: Kun månedslønnede (afl=['0']). Område-filteret (omr) sættes bredt (['1','8']).

# 'trs' afgrænser til de samme koder som Figur 3 (så vi ikke henter hele populationen).
# Selve gruppering/aggregation sker stadig via personalegrupper.py (overenskomst/stilling/klassificering).

payload = {
  "apiKey": "b6caa39c5a6dc520a83ed1624a7bdfdd1935d9fb2bee1d5e2c14dc08ee4ea57ceacca8e9550f5b25ad07b745a612d4b6b5b07bd781baff0908b5891c0beb65c1",
  "table": "Personale-måned",
  "time": [
    {"y1": "2025", "m1": "02"},
    {"y1": "2024", "m1": "02"},
    {"y1": "2023", "m1": "02"}
  ],
  "control": ["overenskomst", "stilling", "klassificering"],
  "data": [
    "hoveder",
    "bhhel",
    "bhdel",
    "bhid1",
    "bhid2",
    "bhid3",
    "bhid4",
    "bpt",
    "gnsalle",
    "gnsmnd",
    "gnsdel"
  ],
  "selection": [
    {
      "name": "Udvalgte population",
      "filters": {
        "trs": [
          # Akademikere
          "272",
          # Læger (066)
          "066", "06609", "06608", "06610", "06618", "06611", "06620", "06617", "06605", "06606",
          # Læger (113)
          "113", "11304", "11310", "11301", "11308",
          # Ledende sygeplejersker
          "301", "30101",
          # Sygeplejersker
          "300", "30001", "30023", "30024", "3002302", "3002402",
          # Jordemødre
          "296", "29608", "29601", "29610", "2960103", "2961003",
          # Fysioterapeuter
          "29605", "2960102", "2961002",
          # Ergoterapeuter
          "29602", "2960101", "2961001",
          # Social- og sundhedsassistenter
          "283", "28305",
          # Sundhedsadministrativt personale
          "055", "05518", "05513", "05514", "05519",
          # Socialpædagoger
          "078", "07801",
          # Omsorgs- og pædagogmedhjælpere m.fl.
          "293",
          # Sygehusportører
          "072",
          # Rengørings- og husassistenter
          "284", "28401", "289", "28901",
          # Erhvervsuddannede serviceassistenter
          "285"
        ],
        "afl": ["0"],
        "omr": ["1", "8"]
      }
    }
  ],
  "options": {
    "totals": True,
    "outputFormat": "json",
    "actions": [],
    "tableName": "Beskæftigelsesgrad",
    "subLimit": 5,
    "modelName": "SIRKA",
    "timeIncreasing": True,
    "tagValues": [
      {"name": "bi1", "dsql": 31},
      {"name": "bi2", "dsql": 27},
      {"name": "bi3", "dsql": 19},
      {"name": "bi1_1", "dsql": 32},
      {"name": "bi2_1", "dsql": 28},
      {"name": "bi3_1", "dsql": 20}
    ]
  },
  "dimension": {
    "viewportHeight": 812,
    "viewportWidth": 1440,
    "xsMaxWidth": 768,
    "smMaxWidth": 992,
    "mdMaxWidth": 1200,
    "CONSTANTS": {"XS": 0, "SM": 1, "MD": 2, "LG": 3, "MAIL": 4}
  }
}

headers = {
    "Accept": "application/json,*/*;q=0.8",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X) Python requests",
}

response = requests.post(url, json=payload, headers=headers, timeout=180)
response.raise_for_status()

rows = response.json()
if isinstance(rows, dict) and "error" in rows:
    raise RuntimeError(rows["error"])
if not isinstance(rows, list):
    raise RuntimeError(f"Forventede en liste med rækker, men fik: {type(rows)}")



In [105]:
import numpy as np
import pandas as pd

import sys
from pathlib import Path
import importlib

# Sørg for at mappen med 'personalegrupper.py' er på Python path
this_dir = Path.cwd()
if str(this_dir) not in sys.path:
    sys.path.insert(0, str(this_dir))

import personalegrupper as pg
pg = importlib.reload(pg)

# Byg en tabel for andel deltid (bhdel/hoveder*100) for samme grupper som Figur 3
if "rows" not in globals() or not isinstance(rows, list):
    raise RuntimeError("Variablen 'rows' mangler. Kør API-cellen først.")

years = [2023, 2024, 2025]
change_col = "Ændring fra 2023 til 2025"

out_rows: list[dict[str, object]] = []
for group_name in pg.ORDER:
    matches = pg.COMBO_GROUPS.get(group_name)
    if matches is None:
        matches = [pg.GROUP_SPECS[group_name]]

    row_out: dict[str, object] = {"Personalegruppe": group_name}
    shares: dict[int, float] = {}
    for y in years:
        ym = f"{y}02"
        personer = pg.sum_matches_for_ym(rows, ym, matches, value_key="hoveder")
        deltid = pg.sum_matches_for_ym(rows, ym, matches, value_key="bhdel")
        share = (deltid / personer) * 100 if personer else np.nan
        shares[y] = float(share) if pd.notna(share) else np.nan
        row_out[y] = shares[y]

    # Ændring i procentpoint (2025 minus 2023)
    if pd.notna(shares[2023]) and pd.notna(shares[2025]):
        row_out[change_col] = shares[2025] - shares[2023]
    else:
        row_out[change_col] = np.nan

    out_rows.append(row_out)

df_share_groups = pd.DataFrame(out_rows).set_index("Personalegruppe")

# Format: 1 decimal og dansk komma
def fmt_1(x) -> str:
    if pd.isna(x):
        return ""
    return f"{x:.1f}".replace(".", ",")

df_formatted = df_share_groups.copy()
for y in years:
    df_formatted[y] = df_share_groups[y].map(fmt_1)
df_formatted[change_col] = df_share_groups[change_col].map(fmt_1)

df_formatted

,2023,2024,2025,Ændring fra 2023 til 2025
Personalegruppe,,,,
Akademikere,"18,1","19,5","20,0","1,9"
Lægelige chefer,"9,0","6,2","7,0","-2,1"
Overlæger,"20,6","22,4","24,0","3,5"
Speciallæger,"15,9","17,9","18,2","2,3"
Uddannelseslæger,"10,0","10,8","11,4","1,4"
Ledende sygeplejersker,"0,0","0,0","3,4","3,4"
Sygeplejersker,"48,1","47,0","45,8","-2,3"
Jordemødre,"39,1","38,1","34,7","-4,4"
Fysioterapeuter,"43,2","45,0","47,1","3,9"
